# 🛡️ Vulnerability Detection - Entraînement du Modèle
Entraînement d'un modèle **CodeBERT** pour la détection de vulnérabilités dans le code source.

## 1. Imports et vérification GPU

In [ ]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Vérification GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Chargement du dataset tokenisé

In [ ]:
dataset = load_from_disk("Defectors_Tokenized_Final")
print(f"Train: {len(dataset['train'])} exemples")
print(f"Validation: {len(dataset['validation'])} exemples")
print(f"Test: {len(dataset['test'])} exemples")

# Garder seulement les colonnes nécessaires
dataset = dataset.remove_columns(["code"])
dataset.set_format("torch")

## 3. Chargement du modèle CodeBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2
)
model.to(device)
print(f"Modèle chargé sur {device}")

## 4. Métriques d'évaluation

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## 5. Configuration de l'entraînement

In [ ]:
training_args = TrainingArguments(
    output_dir="./vulnerability_model_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

## 6. Lancer l'entraînement

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

print("Début de l'entraînement...")
trainer.train()

## 7. Évaluation sur le test set

In [ ]:
results = trainer.evaluate(dataset["test"])
print("Résultats sur le test set:")
for key, value in results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## 8. Sauvegarde du modèle final

In [ ]:
trainer.save_model("./vulnerability_model_final")
print("Modèle sauvegardé dans ./vulnerability_model_final")
print("Entraînement terminé !")